In [ ]:
import sys
sys.path.append('..')

from constants import *

print("Registering temporary views from Parquet files...")
for table_name in os.listdir(PARQUET_DIR):
    table_path = os.path.join(PARQUET_DIR, table_name)
    if os.path.isdir(table_path):
        # Read parquet dataset
        df = SPARK_SESSION.read.parquet(table_path)
        # Register with its original table name (e.g., "store_sales") for query resolution
        df.createOrReplaceTempView(table_name)
print("All tables are ready for SQL queries!")

In [ ]:
SPARK_SESSION.sql("show tables;").show(n=50, truncate=False)

In [ ]:
query = """with customer_total_return as
(select sr_customer_sk as ctr_customer_sk
,sr_store_sk as ctr_store_sk
,sum(SR_FEE) as ctr_total_return
from store_returns
,date_dim
where sr_returned_date_sk = d_date_sk
and d_year =2000
group by sr_customer_sk
,sr_store_sk)
 select  c_customer_id
from customer_total_return ctr1
,store
,customer
where ctr1.ctr_total_return > (select avg(ctr_total_return)*1.2
from customer_total_return ctr2
where ctr1.ctr_store_sk = ctr2.ctr_store_sk)
and s_store_sk = ctr1.ctr_store_sk
and s_state = 'TN'
and ctr1.ctr_customer_sk = c_customer_sk
order by c_customer_id
limit 100;
"""

res = SPARK_SESSION.sql(query)

In [ ]:
n = res.count()
print(f"Number of rows returned: {n}")
print(res.show(n=n, truncate=False))